# Lab 11: Implement RAG-Based LLM Project Using Vector DB


In [1]:
!pip install langchain langchain-groq langchain-community chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71

## Step 1: Prepare and Chunk Documents


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = [
    """Generative AI refers to AI systems that can create new content such as text, images, audio,
    and video. Examples include GPT-4 for text generation, DALL-E for image creation, and Sora for
    video generation. These models are trained on large datasets and learn to generate new samples
    that resemble the training data.""",

    """Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with
    text generation. Instead of relying solely on the LLM's training knowledge, RAG retrieves
    relevant documents from an external knowledge base and provides them as context to the LLM.
    This reduces hallucination and allows models to answer questions about recent or private data.""",

    """Vector databases store data as high-dimensional vectors (embeddings) and enable fast
    similarity search. Popular vector databases include Pinecone, Weaviate, Qdrant, Milvus, and
    ChromaDB. They are essential for RAG systems as they efficiently find the most semantically
    similar documents to a given query.""",

    """LangChain is a framework for building LLM applications. It provides components for
    document loading, text splitting, embedding, vector storage, and chain creation.
    LangChain supports many LLM providers and vector databases, making it ideal for RAG pipelines.""",

    """Embeddings are numerical vector representations of text that capture semantic meaning.
    Similar texts have embeddings that are close together in vector space. Models like
    sentence-transformers/all-MiniLM-L6-v2 create 384-dimensional embeddings and are widely used
    for semantic search and RAG applications."""
]

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.create_documents(documents)

print(f"Total documents: {len(documents)}")
print(f"Total chunks after splitting: {len(chunks)}")
print("\nFirst chunk preview:")
print(chunks[0].page_content)


Total documents: 5
Total chunks after splitting: 9

First chunk preview:
Generative AI refers to AI systems that can create new content such as text, images, audio, 
    and video. Examples include GPT-4 for text generation, DALL-E for image creation, and Sora for 
    video generation. These models are trained on large datasets and learn to generate new samples


## Step 2: Create Embeddings and Store in ChromaDB


In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize embedding model (free, runs locally)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create vector store and add documents
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print(f"Vector store created with {vectorstore._collection.count()} vectors")

/tmp/ipykernel_7828/3718361965.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store created with 9 vectors


## Step 3: Test Similarity Search


In [5]:
# Test retrieval
query = "What is RAG and why is it useful?"
retrieved_docs = vectorstore.similarity_search(query, k=2)

print(f"Query: {query}")
print(f"\nTop {len(retrieved_docs)} retrieved chunks:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Chunk {i} ---")
    print(doc.page_content)

Query: What is RAG and why is it useful?

Top 2 retrieved chunks:

--- Chunk 1 ---
Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with 
    text generation. Instead of relying solely on the LLM's training knowledge, RAG retrieves 
    relevant documents from an external knowledge base and provides them as context to the LLM.

--- Chunk 2 ---
for semantic search and RAG applications.


## Step 4: Build the RAG Chain


In [8]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

groq_api_key = "yourapikey"
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, api_key=groq_api_key)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# RAG prompt
rag_prompt = ChatPromptTemplate.from_template("""
You are an AI assistant. Answer the question using ONLY the provided context.
If the context doesn't contain enough information, say so clearly.

Context:
{context}

Question: {question}

Answer:
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built successfully.")


RAG chain built successfully.


## Step 5: Question Answering with RAG


In [9]:
test_questions = [
    "What is Retrieval-Augmented Generation?",
    "What are some popular vector databases?",
    "How do embeddings work?",
    "What is the capital of France?"  # This should return 'not found'
]

for question in test_questions:
    print(f"Q: {question}")
    answer = rag_chain.invoke(question)
    print(f"A: {answer}")
    print("-" * 60)

Q: What is Retrieval-Augmented Generation?
A: Retrieval-Augmented Generation (RAG) is a technique that combines information retrieval with text generation.
------------------------------------------------------------
Q: What are some popular vector databases?
A: Some popular vector databases include Pinecone, Weaviate, Qdrant, Milvus, and ChromaDB.
------------------------------------------------------------
Q: How do embeddings work?
A: Embeddings are numerical vector representations of text that capture semantic meaning. Similar texts have embeddings that are close together in vector space. This means that embeddings work by converting text into numerical vectors, allowing for the comparison and similarity search of text data in a high-dimensional vector space.
------------------------------------------------------------
Q: What is the capital of France?
A: There is no information in the provided context about the capital of France.
---------------------------------------------------